# Website Video Generator — Google Colab MVP

Masukkan URL website publik, pilih template, lalu buat preview dan video MP4.

**Catatan:** GPU opsional. HyperFrames juga opsional; notebook memiliki renderer FFmpeg fallback agar MVP tetap dapat berjalan.

In [ ]:
import os, sys, subprocess, shutil, platform, json, re, time, math
from pathlib import Path

PROJECT = Path('/content/website_video_generator')
for name in ['captures','output','temp','compositions']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)

packages = ['gradio>=4.0','playwright','requests','beautifulsoup4','Pillow','imageio-ffmpeg','psutil']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'], check=True)
print('Project:', PROJECT)
print('FFmpeg:', shutil.which('ffmpeg') or 'imageio-ffmpeg fallback')
print('Optional HyperFrames install can be attempted separately; it is not required.')

In [ ]:
import psutil
print('Python:', sys.version)
print('Platform:', platform.platform())
print('RAM GB:', round(psutil.virtual_memory().total/1024**3, 2))
print('Disk free GB:', round(shutil.disk_usage('/content').free/1024**3, 2))
gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'Tidak tersedia / CPU runtime')

In [ ]:
import asyncio, requests, html
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from PIL import Image, ImageDraw, ImageFont

UA = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36'

def normalize_url(value):
    value = (value or '').strip()
    if not re.match(r'^https?://', value, re.I): value = 'https://' + value
    parsed = urlparse(value)
    if not parsed.netloc: raise ValueError('URL tidak valid.')
    return value

def validate_url(value):
    url = normalize_url(value)
    response = requests.get(url, headers={'User-Agent': UA}, timeout=20, allow_redirects=True)
    if response.status_code >= 400: raise ValueError(f'HTTP {response.status_code}: website tidak dapat diakses.')
    return response.url

async def capture(url, folder, width, height, steps=5):
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=['--no-sandbox','--disable-dev-shm-usage'])
        page = await browser.new_page(viewport={'width': width, 'height': height}, user_agent=UA)
        await page.goto(url, wait_until='domcontentloaded', timeout=60000)
        await page.wait_for_timeout(2500)
        page_height = await page.evaluate('Math.max(document.body.scrollHeight, document.documentElement.scrollHeight)')
        max_scroll = max(0, page_height-height)
        positions = [0] if max_scroll == 0 else [round(max_scroll*i/max(steps-1,1)) for i in range(steps)]
        shots = []
        for i, pos in enumerate(positions):
            await page.evaluate('(y) => window.scrollTo(0,y)', pos)
            await page.wait_for_timeout(700)
            path = folder/f'capture_{i:02d}.png'
            await page.screenshot(path=str(path), full_page=False)
            shots.append({'path': str(path), 'scroll_y': pos, 'index': i})
        await browser.close()
    return {'url': url, 'page_height': page_height, 'screenshots': shots}

In [ ]:
def analyze(url):
    try:
        text = requests.get(url, headers={'User-Agent': UA}, timeout=20).text
        soup = BeautifulSoup(text, 'lxml')
        headings = [x.get_text(' ', strip=True) for x in soup.find_all(['h1','h2','h3'])]
        blob = ' '.join(headings).lower()
        labels = []
        for key, words in {'hero':['hero','welcome','build','business'], 'features':['feature','solution','everything'], 'product':['product','platform','dashboard'], 'testimonial':['testimonial','customer','stories'], 'cta':['start','trial','contact','learn more']}.items():
            if any(w in blob for w in words): labels.append(key)
        return {'headings': headings[:12], 'sections': labels}
    except Exception as e:
        return {'headings': [], 'sections': [], 'warning': str(e)}

def storyboard(meta, duration, mode, title, brand, cta):
    shots = meta['screenshots']
    opening = min(3, duration*0.12); closing = min(3, duration*0.12)
    middle = max(1, duration-opening-closing) / max(1, len(shots))
    scenes = [{'name':'opening','duration':opening,'image':None,'text':title or brand or 'Website Showcase'}]
    for i, shot in enumerate(shots):
        label = 'hero' if i == 0 else ('cta' if i == len(shots)-1 else f'section_{i}')
        text = cta if label == 'cta' else (title if label == 'hero' else '')
        scenes.append({'name':label,'duration':middle,'image':shot['path'],'text':text})
    scenes.append({'name':'closing','duration':closing,'image':None,'text':cta or brand or 'Learn more'})
    return {'mode': mode, 'scenes': scenes, 'total': sum(x['duration'] for x in scenes)}

In [ ]:
def make_frame(scene, size, bg, brand, zoom=1.0, mockup=True):
    w,h = size; canvas = Image.new('RGB', size, bg); draw = ImageDraw.Draw(canvas)
    if scene.get('image'):
        img = Image.open(scene['image']).convert('RGB')
        target_w = int(w*.84) if mockup else int(w*.96); target_h = int(h*.62)
        img.thumbnail((target_w, target_h))
        x=(w-img.width)//2; y=int(h*.16)
        if mockup:
            draw.rounded_rectangle((x-8,y-38,x+img.width+8,y+img.height+8), radius=16, fill='#1f2937')
            for dot, color in enumerate(['#ef4444','#f59e0b','#22c55e']): draw.ellipse((x+10+dot*18,y-27,x+20+dot*18,y-17), fill=color)
            y += 4
        canvas.paste(img,(x,y))
    if brand: draw.text((int(w*.07),int(h*.06)), brand, fill='white')
    text = scene.get('text','')
    if text: draw.text((int(w*.08),int(h*.84)), text[:90], fill='white')
    return canvas

def render_video(sb, output, width, height, fps=24, bg='#0f172a', brand='', mockup=True):
    frame_dir = PROJECT/'temp'/'frames'; frame_dir.mkdir(parents=True, exist_ok=True)
    for old in frame_dir.glob('*.png'): old.unlink()
    index=0
    for scene in sb['scenes']:
        count=max(1, round(scene['duration']*fps))
        for _ in range(count):
            make_frame(scene,(width,height),bg,brand,mockup=mockup).save(frame_dir/f'{index:06d}.png'); index+=1
    output=Path(output); output.parent.mkdir(parents=True, exist_ok=True)
    ffmpeg=shutil.which('ffmpeg')
    if not ffmpeg:
        import imageio_ffmpeg; ffmpeg=imageio_ffmpeg.get_ffmpeg_exe()
    cmd=[ffmpeg,'-y','-framerate',str(fps),'-i',str(frame_dir/'%06d.png'),'-c:v','libx264','-pix_fmt','yuv420p','-movflags','+faststart',str(output)]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return str(output)

In [ ]:
import gradio as gr

def generate(url, mode, orientation, duration, resolution, title, brand, cta, bg, speed, mockup, zoom, action):
    try:
        final_url = validate_url(url)
        if orientation == 'Landscape 16:9': base=(1280,720)
        else: base=(720,1280)
        if resolution == '720p': base=(720,405) if orientation.startswith('Landscape') else (405,720)
        capture_dir = PROJECT/'captures'/str(int(time.time()))
        meta = asyncio.run(capture(final_url, capture_dir, 1280, 720, 5))
        analysis = analyze(final_url)
        sb = storyboard(meta, int(duration), mode, title, brand, cta)
        preview = make_frame(sb['scenes'][1] if len(sb['scenes'])>1 else sb['scenes'][0], base, bg, brand, mockup=mockup)
        if action == 'Generate Preview': return preview, None, json.dumps({'url':final_url,'analysis':analysis,'storyboard':sb}, indent=2, ensure_ascii=False)
        out = PROJECT/'output'/f'website_showcase_{int(time.time())}.mp4'
        video = render_video(sb, out, *base, bg=bg, brand=brand, mockup=mockup)
        return preview, video, json.dumps({'url':final_url,'analysis':analysis,'storyboard':sb}, indent=2, ensure_ascii=False)
    except Exception as e:
        return None, None, 'ERROR: '+str(e)

with gr.Blocks(title='Website Video Generator') as demo:
    gr.Markdown('# Website Video Generator')
    with gr.Row():
        url=gr.Textbox(label='URL website', placeholder='https://example.com')
        mode=gr.Dropdown(['Website Tour','Cinematic Showcase','Hybrid Showcase'], value='Hybrid Showcase', label='Mode')
    with gr.Row():
        orientation=gr.Dropdown(['Landscape 16:9','Portrait 9:16'], value='Portrait 9:16', label='Orientasi')
        duration=gr.Dropdown([15,30,45,60], value=30, label='Durasi (detik)')
        resolution=gr.Dropdown(['720p','1080p'], value='720p', label='Resolusi')
    with gr.Row():
        title=gr.Textbox(label='Judul'); brand=gr.Textbox(label='Brand'); cta=gr.Textbox(label='CTA')
    with gr.Row():
        bg=gr.ColorPicker(value='#0f172a', label='Background'); speed=gr.Dropdown(['Slow','Medium','Fast'], value='Medium', label='Kecepatan scroll'); mockup=gr.Checkbox(value=True,label='Browser mockup'); zoom=gr.Checkbox(value=True,label='Efek zoom')
    action=gr.Radio(['Generate Preview','Render Video'], value='Generate Preview', label='Aksi')
    run=gr.Button('Jalankan', variant='primary')
    with gr.Row(): preview=gr.Image(label='Preview'); video=gr.Video(label='Video MP4')
    details=gr.Code(label='Detail storyboard / status', language='json')
    run.click(generate, [url,mode,orientation,duration,resolution,title,brand,cta,bg,speed,mockup,zoom,action], [preview,video,details])

demo.launch(share=True, debug=True)